# Utils Notebook

This notebook centralizes shared logic for both Part A and Part B.
Use these classes in downstream notebooks instead of duplicating helper functions.


## 1) Imports and Global Constants

This section loads all dependencies used across data loading, plotting, model setup, and training.
It also defines shared ImageNet normalization constants.


In [ ]:
import os
import sys
import subprocess
import shutil
import zipfile
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets, models, transforms

try:
    from ignite.engine import Engine, Events
    from ignite.handlers import EarlyStopping
    from ignite.metrics import Accuracy, Loss as IgniteLoss
    from ignite.contrib.handlers import ProgressBar
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytorch-ignite", "-q"])
    from ignite.engine import Engine, Events
    from ignite.handlers import EarlyStopping
    from ignite.metrics import Accuracy, Loss as IgniteLoss
    from ignite.contrib.handlers import ProgressBar

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


## 2) Setup Utilities Class

`SetupUtils` handles reproducibility and device selection.
Instantiate once at the top of each notebook.


In [ ]:
class SetupUtils:
    """Utilities for reproducibility and hardware setup."""

    @staticmethod
    def set_global_seed(seed: int = 42) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    @staticmethod
    def get_device() -> torch.device:
        device = (
            torch.device("cuda") if torch.cuda.is_available() else
            torch.device("mps") if torch.backends.mps.is_available() else
            torch.device("cpu")
        )
        if device.type == "cuda":
            torch.backends.cudnn.benchmark = True
        return device


## 3) Data Pipeline Utilities Class

`DataPipelineUtils` encapsulates:
- transform creation for Part A and Part B,
- official split parsing,
- stratified train/val split,
- dataloader construction.

Both notebooks should use the same stratified split method from this class.


In [ ]:
class DataPipelineUtils:
    """Shared data transforms and split/loader orchestration."""

    def __init__(self, img_size: int = 224, imagenet_mean=None, imagenet_std=None):
        self.img_size = img_size
        self.imagenet_mean = imagenet_mean or IMAGENET_MEAN
        self.imagenet_std = imagenet_std or IMAGENET_STD

    def build_transforms(self):
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(self.img_size, scale=(0.5, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(20),
            transforms.ToTensor(),
            transforms.Normalize(self.imagenet_mean, self.imagenet_std),
        ])
        eval_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize(self.imagenet_mean, self.imagenet_std),
        ])
        return train_transform, eval_transform

    @staticmethod
    def read_split_file(filepath: Path):
        with open(filepath, "r") as f:
            return set(line.strip() for line in f.readlines())

    @staticmethod
    def map_official_split_indices(samples, train_keys, test_keys):
        train_indices, test_indices = [], []
        for idx, (file_path, _) in enumerate(samples):
            p = Path(file_path)
            key = f"{p.parent.name}/{p.stem}"
            if key in train_keys:
                train_indices.append(idx)
            elif key in test_keys:
                test_indices.append(idx)
        return train_indices, test_indices

    @staticmethod
    def stratified_train_val_split(train_indices, samples, val_fraction: float, seed: int = 42):
        rng = random.Random(seed)
        indices_by_class = {}
        for idx in train_indices:
            class_id = samples[idx][1]
            indices_by_class.setdefault(class_id, []).append(idx)

        train_split_indices, val_indices = [], []
        for cls_indices in indices_by_class.values():
            cls_indices = cls_indices.copy()
            rng.shuffle(cls_indices)
            val_count = int(val_fraction * len(cls_indices))
            val_indices.extend(cls_indices[:val_count])
            train_split_indices.extend(cls_indices[val_count:])

        rng.shuffle(train_split_indices)
        rng.shuffle(val_indices)
        return train_split_indices, val_indices

    def build_food101_splits_and_loaders(
        self,
        img_dir: Path,
        meta_dir: Path,
        train_transform,
        eval_transform,
        batch_size: int,
        num_workers: int,
        val_fraction: float,
        seed: int = 42,
    ):
        full_train_dataset = datasets.ImageFolder(root=img_dir, transform=train_transform)
        full_eval_dataset = datasets.ImageFolder(root=img_dir, transform=eval_transform)

        train_keys = self.read_split_file(meta_dir / "train.txt")
        test_keys = self.read_split_file(meta_dir / "test.txt")

        train_indices, test_indices = self.map_official_split_indices(
            full_train_dataset.samples, train_keys, test_keys
        )

        train_indices, val_indices = self.stratified_train_val_split(
            train_indices, full_train_dataset.samples, val_fraction, seed
        )

        train_dataset = Subset(full_train_dataset, train_indices)
        val_dataset = Subset(full_eval_dataset, val_indices)
        test_dataset = Subset(full_eval_dataset, test_indices)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

        return {
            "full_train_dataset": full_train_dataset,
            "full_eval_dataset": full_eval_dataset,
            "train_indices": train_indices,
            "val_indices": val_indices,
            "test_indices": test_indices,
            "train_dataset": train_dataset,
            "val_dataset": val_dataset,
            "test_dataset": test_dataset,
            "train_loader": train_loader,
            "val_loader": val_loader,
            "test_loader": test_loader,
            "classes": full_eval_dataset.classes,
        }


## 4) Visualization Utilities Class

`TrainingVisualizer` contains all plotting logic:
- random dataset sample panel,
- class distribution,
- train/val curves,
- test accuracy bars,
- confusion matrices,
- augmentation previews.


In [ ]:
class TrainingVisualizer:
    """Shared plotting utilities for transfer learning and fine-tuning experiments."""
    _TRAIN_COLOR = "steelblue"
    _VAL_COLOR = "darkorange"

    def plot_random_dataset_samples(self, classes: list, img_dir: Path, n_samples: int = 12) -> None:
        sample_classes = random.sample(classes, n_samples)
        rows, cols = 3, 4
        fig, axes = plt.subplots(rows, cols, figsize=(14, 10))
        for ax, cls in zip(axes.flatten(), sample_classes):
            class_dir = img_dir / cls
            img_file = random.choice(list(class_dir.glob("*.jpg")))
            img = mpimg.imread(img_file)
            ax.imshow(img)
            ax.set_title(cls.replace("_", " ").title(), fontsize=9)
            ax.axis("off")

        plt.suptitle("Food101 — Random sample of 12 classes", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

    def _get_labels_from_dataset(self, dataset):
        if isinstance(dataset, Subset):
            base_labels = self._get_labels_from_dataset(dataset.dataset)
            return [base_labels[i] for i in dataset.indices]
        if isinstance(dataset, ConcatDataset):
            labels = []
            for ds in dataset.datasets:
                labels.extend(self._get_labels_from_dataset(ds))
            return labels
        if hasattr(dataset, "targets"):
            return dataset.targets
        if hasattr(dataset, "_labels"):
            return dataset._labels
        if hasattr(dataset, "labels"):
            return dataset.labels
        return [dataset[i][1] for i in range(len(dataset))]

    def plot_class_distribution(self, datasets_dict: dict, class_names: list = None, figsize_per_row: tuple = (16, 4)) -> None:
        counts_dict = {}
        for name, dataset in datasets_dict.items():
            labels = self._get_labels_from_dataset(dataset)
            if len(labels) > 0 and torch.is_tensor(labels[0]):
                labels = [l.item() for l in labels]
            counts_dict[name] = Counter(labels)

        all_classes_idx = sorted(list(set().union(*(c.keys() for c in counts_dict.values()))))
        labels_shown = [class_names[i].replace("_", " ") if class_names else str(i) for i in all_classes_idx]

        x = np.arange(len(all_classes_idx))
        n_splits = len(datasets_dict)
        split_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

        fig, axes = plt.subplots(n_splits, 1, figsize=(figsize_per_row[0], figsize_per_row[1] * n_splits))
        if n_splits == 1:
            axes = [axes]

        for i, ((name, counter), ax) in enumerate(zip(counts_dict.items(), axes)):
            vals = [counter.get(c, 0) for c in all_classes_idx]
            ax.bar(x, vals, width=0.8, color=split_colors[i % len(split_colors)])
            ax.set_ylabel("Samples")
            ax.set_title(f"{name} Dataset", fontweight="bold")
            ax.set_xticks(x)
            if i == n_splits - 1:
                ax.set_xticklabels(labels_shown, rotation=90, ha="center", fontsize=8)
            else:
                ax.set_xticklabels([])
        plt.tight_layout()
        plt.show()

    def plot_loss(self, histories: dict, figsize_per: tuple = (6, 4)) -> None:
        self._plot_metric(histories, metric="loss", ylabel="Cross-Entropy Loss", title="Loss Curves", figsize_per=figsize_per)

    def plot_accuracy(self, histories: dict, figsize_per: tuple = (6, 4)) -> None:
        self._plot_metric(histories, metric="acc", ylabel="Accuracy", title="Accuracy Curves", figsize_per=figsize_per)

    def _plot_metric(self, histories: dict, metric: str, ylabel: str, title: str, figsize_per: tuple) -> None:
        n = len(histories)
        fig, axes = plt.subplots(1, n, figsize=(figsize_per[0] * n, figsize_per[1]), sharey=(metric == "acc"))
        if n == 1:
            axes = [axes]

        for ax, (name, hist) in zip(axes, histories.items()):
            epochs = range(1, len(hist[f"train_{metric}"]) + 1)
            ax.plot(epochs, hist[f"train_{metric}"], label="Train", color=self._TRAIN_COLOR, marker="o", ms=4)
            ax.plot(epochs, hist[f"val_{metric}"], label="Val", color=self._VAL_COLOR, marker="s", ms=4)
            ax.set_title(name)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(ylabel)
            ax.legend()
            ax.grid(True, linestyle="--", alpha=0.5)
        plt.suptitle(title, fontweight="bold")
        plt.tight_layout()
        plt.show()


    def _resolve_device(self, device="cuda"):
        if device == "cuda" and not torch.cuda.is_available():
            device = "mps" if torch.backends.mps.is_available() else "cpu"
        elif device == "mps" and not torch.backends.mps.is_available():
            device = "cpu"
        return torch.device(device)

    def _compute_confusion_matrix(self, model, loader, class_names, device="cuda"):
        from sklearn.metrics import confusion_matrix

        device = self._resolve_device(device)
        model.to(device).eval()

        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in tqdm(loader, leave=False, desc="Confusion Matrix"):
                outputs = model(images.to(device))
                all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
                all_labels.extend(labels.numpy())

        return confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))

    def _top_confusion_pairs(self, cm, class_names, top_k=1):
        directed = []
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                if i == j:
                    continue
                count = int(cm[i, j])
                if count > 0:
                    directed.append((count, i, j))
        directed.sort(reverse=True)

        bidirectional = []
        for i in range(len(class_names)):
            for j in range(i + 1, len(class_names)):
                count = int(cm[i, j] + cm[j, i])
                if count > 0:
                    bidirectional.append((count, i, j))
        bidirectional.sort(reverse=True)

        return directed[:top_k], bidirectional[:top_k]

    def plot_confusion_matrix(
        self,
        model,
        loader,
        class_names,
        top_n=20,
        figsize=(14, 12),
        device="cuda",
        report_top_pairs=False,
        top_k_pairs=1,
    ):
        cm = self._compute_confusion_matrix(model, loader, class_names, device=device)

        def _show_cm(cm_slice, labels, title):
            denom = cm_slice.sum(axis=1, keepdims=True).clip(min=1)
            plt.figure(figsize=figsize)
            sns.heatmap(cm_slice / denom, annot=(top_n is None or top_n <= 30), fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
            plt.title(title, fontweight="bold")
            plt.xticks(rotation=45, ha="right")
            plt.show()

        if top_n and top_n < len(class_names):
            errors = cm.sum(axis=1) - np.diag(cm)
            most_idx = np.argsort(errors)[-top_n:][::-1]
            least_idx = np.argsort(errors)[:top_n]
            _show_cm(cm[np.ix_(most_idx, most_idx)], [class_names[i] for i in most_idx], f"Top {top_n} Most Confused")
            _show_cm(cm[np.ix_(least_idx, least_idx)], [class_names[i] for i in least_idx], f"Top {top_n} Least Confused")
        else:
            _show_cm(cm, class_names, "Confusion Matrix")

        if report_top_pairs:
            directed, bidirectional = self._top_confusion_pairs(cm, class_names, top_k=max(1, int(top_k_pairs)))

            if directed:
                print("Top directed confusion(s):")
                for count, i, j in directed:
                    src = class_names[i].replace("_", " ")
                    dst = class_names[j].replace("_", " ")
                    print(f"- {src} -> {dst}: {count} images")

            if bidirectional:
                print("Top confusion pair(s) (both directions):")
                for count, i, j in bidirectional:
                    a = class_names[i].replace("_", " ")
                    b = class_names[j].replace("_", " ")
                    print(f"- {a} <-> {b}: {count} images")

    def plot_augmentation_samples(self, image_path: str, img_size: int = 224, mean: list = None, std: list = None) -> None:
        mean = mean or IMAGENET_MEAN
        std = std or IMAGENET_STD
        try:
            orig_img = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image: {e}")
            return

        steps = [
            ("Original", None),
            ("Resize Only", transforms.Resize((img_size, img_size))),
            ("RandomResizedCrop", transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0))),
            ("HorizontalFlip", transforms.Compose([transforms.Resize((img_size, img_size)), transforms.RandomHorizontalFlip(p=1.0)])),
            ("ColorJitter", transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ColorJitter(0.2, 0.2, 0.2, 0.1)])),
            ("RandomRotation", transforms.Compose([transforms.Resize((img_size, img_size)), transforms.RandomRotation(20)])),
            ("Normalized Tensor", transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), transforms.Normalize(mean, std)])),
        ]

        fig, axes = plt.subplots(1, len(steps), figsize=(25, 5))
        for i, (name, transform) in enumerate(steps):
            img = transform(orig_img) if transform else orig_img
            ax = axes[i]
            if torch.is_tensor(img):
                img_disp = img.numpy().transpose((1, 2, 0))
                img_disp = (np.array(std) * img_disp) + np.array(mean)
                ax.imshow(np.clip(img_disp, 0, 1))
            else:
                ax.imshow(img)
            ax.set_title(name, fontsize=12, fontweight="bold")
            ax.axis("off")

        plt.suptitle("Independent Data Augmentation Samples", fontsize=16, fontweight="bold", y=1.08)
        plt.tight_layout()
        plt.show()


## 5) Model Utilities Class

`ModelUtils` stores model architecture operations for both parts:
- custom head building,
- freezing and parameter counting,
- fine-tuning block selection,
- experiment model preparation from checkpoints.


In [ ]:
class ModelUtils:
    """Shared model architecture and fine-tuning helpers."""

    def __init__(self, device: torch.device, num_classes: int = 101):
        self.device = device
        self.num_classes = num_classes

    def build_custom_head(self, in_features: int, num_classes: int = None) -> nn.Sequential:
        n_classes = num_classes if num_classes is not None else self.num_classes
        return nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, n_classes),
        )


    @staticmethod
    def count_params(model: nn.Module):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return total, trainable

    def build_model(self, model_name: str, weights=None) -> nn.Module:
        if model_name == "googlenet":
            m = models.googlenet(weights=weights)
            for p in m.parameters():
                p.requires_grad = False
            m.aux_logits = False
            m.fc = self.build_custom_head(m.fc.in_features, self.num_classes)
        elif model_name == "mobilenet":
            m = models.mobilenet_v3_large(weights=weights)
            for p in m.parameters():
                p.requires_grad = False
            in_feat = m.classifier[0].in_features
            m.classifier = self.build_custom_head(in_feat, self.num_classes)
        elif model_name == "resnet":
            m = models.resnet50(weights=weights)
            for p in m.parameters():
                p.requires_grad = False
            m.fc = self.build_custom_head(m.fc.in_features, self.num_classes)
        else:
            raise ValueError(f"Unknown model: {model_name}")
        return m.to(self.device)

    @staticmethod
    def get_backbone_children(model: nn.Module, model_name: str):
        if model_name == "mobilenet":
            return [(f"features[{idx}]", block) for idx, block in enumerate(model.features)]
        if model_name == "resnet":
            return [("layer1", model.layer1), ("layer2", model.layer2), ("layer3", model.layer3), ("layer4", model.layer4)]
        if model_name == "googlenet":
            blocks = [
                "conv1", "maxpool1", "conv2", "conv3", "maxpool2",
                "inception3a", "inception3b", "maxpool3",
                "inception4a", "inception4b", "inception4c", "inception4d", "inception4e", "maxpool4",
                "inception5a", "inception5b",
            ]
            return [(name, getattr(model, name)) for name in blocks if hasattr(model, name)]
        raise ValueError(f"Unknown model: {model_name}")

    def unfreeze_top_n_layers(self, model: nn.Module, model_name: str, n: int) -> None:
        if n == 0:
            print("Backbone fully frozen — training head only.")
            return

        children = self.get_backbone_children(model, model_name)
        if not children:
            print("No backbone blocks found to unfreeze.")
            return

        if n > 0 and n > len(children):
            print(f"Requested {n} blocks but model has {len(children)}; unfreezing all.")
            n = len(children)

        to_unfreeze = children if n == -1 else children[-n:]

        for name, module in to_unfreeze:
            for p in module.parameters():
                p.requires_grad = True
            print(f"  Unfrozen backbone block: {name}")

        frozen = [name for name, _ in children if name not in {nm for nm, _ in to_unfreeze}]
        print(f"  Still frozen ({len(frozen)}): {frozen}")

    @staticmethod
    def count_trainable(model: nn.Module):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.1f} %)")

    def prepare_experiment_model(self, model_name: str, n_unfrozen: int, checkpoint_path: Path) -> nn.Module:
        model = self.build_model(model_name)
        model.load_state_dict(torch.load(checkpoint_path, map_location=self.device))

        print(f"=== Experiment: unfreeze last {n_unfrozen} backbone block(s) ===")
        self.unfreeze_top_n_layers(model, model_name, n_unfrozen)

        head_attr = "fc" if model_name in ("googlenet", "resnet") else "classifier"
        for p in getattr(model, head_attr).parameters():
            p.requires_grad = True

        self.count_trainable(model)
        return model


## 6) Training Utilities Class

`TrainingUtils` keeps fine-tuning loop logic and checkpoint handling.
Instantiate this class in Part B after path and hyperparameter configuration is defined.


In [ ]:
class TrainingUtils:
    """Shared training engines for transfer learning and fine-tuning."""

    def __init__(
        self,
        device: torch.device,
        checkpoint_dir: Path,
        head_lr: float = 1e-3,
        backbone_lr: float = 1e-4,
        default_epochs: int = 20,
    ):
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.head_lr = head_lr
        self.backbone_lr = backbone_lr
        self.default_epochs = default_epochs


    def _run_training_loop(
        self,
        model,
        run_label: str,
        train_loader,
        val_loader,
        optimiser,
        scheduler,
        epochs: int,
        early_stop_patience: int,
        save_last_checkpoint: bool = False,
        show_progress_bar: bool = False,
        print_run_header: str = None,
        print_lr: bool = False,
        return_best_acc: bool = False,
    ):
        """Single shared Ignite training loop used by Part A and Part B."""
        criterion = nn.CrossEntropyLoss()
        use_amp = self.device.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        best_val_acc = 0.0

        def _train_step(engine, batch):
            model.train()
            images, labels = batch
            images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
            optimiser.zero_grad()

            with torch.amp.autocast(device_type=self.device.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimiser)
            scaler.update()

            engine.state.train_loss_sum = getattr(engine.state, "train_loss_sum", 0.0) + loss.item() * images.size(0)
            engine.state.train_correct = getattr(engine.state, "train_correct", 0) + outputs.argmax(1).eq(labels).sum().item()
            engine.state.train_total = getattr(engine.state, "train_total", 0) + labels.size(0)
            return outputs, labels

        def _eval_step(engine, batch):
            model.eval()
            with torch.no_grad():
                images, labels = batch
                images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
                with torch.amp.autocast(device_type=self.device.type, enabled=use_amp):
                    outputs = model(images)
                return outputs, labels

        trainer = Engine(_train_step)
        evaluator = Engine(_eval_step)

        IgniteLoss(criterion).attach(evaluator, "val_loss")
        Accuracy().attach(evaluator, "val_acc")

        if show_progress_bar:
            trainer_pbar = ProgressBar(
                persist=False,
                bar_format="Training |{bar:40}| {percentage:3.0f}% [{elapsed}<{remaining}]",
            )
            trainer_pbar.attach(trainer)
        else:
            trainer_pbar = None

        early_stop_handler = EarlyStopping(
            patience=early_stop_patience,
            score_function=lambda e: -e.state.metrics["val_loss"],
            trainer=trainer,
        )
        evaluator.add_event_handler(Events.COMPLETED, early_stop_handler)

        if print_run_header:
            print(f"\n🚀 Training: {print_run_header} | Device: {self.device} (AMP={use_amp})")

        @trainer.on(Events.EPOCH_STARTED)
        def _reset_train_metrics(engine):
            engine.state.train_loss_sum = 0.0
            engine.state.train_correct = 0
            engine.state.train_total = 0

        @trainer.on(Events.EPOCH_COMPLETED)
        def _on_epoch_end(engine):
            nonlocal best_val_acc
            epoch = engine.state.epoch

            tr_loss = engine.state.train_loss_sum / engine.state.train_total
            tr_acc = engine.state.train_correct / engine.state.train_total

            evaluator.run(val_loader)
            val_loss = evaluator.state.metrics["val_loss"]
            val_acc = evaluator.state.metrics["val_acc"]

            scheduler.step(val_loss)

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(tr_acc)
            history["val_acc"].append(val_acc)

            best_status = ""
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_path = self.checkpoint_dir / f"{run_label}_best.pth"
                torch.save(model.state_dict(), best_path)
                best_status = "⭐ NEW BEST"

            if save_last_checkpoint:
                torch.save(
                    {
                        "epoch": epoch,
                        "run_label": run_label,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimiser.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "best_val_acc": best_val_acc,
                        "history": {k: list(v) for k, v in history.items()},
                    },
                    self.checkpoint_dir / f"{run_label}_checkpoint_last.pt",
                )

            if trainer_pbar is not None:
                trainer_pbar.log_message(
                    f"Epoch {epoch:>2}/{epochs} ─ "
                    f"Tr Loss: {tr_loss:.4f} | Tr Acc: {tr_acc:.4f} ── "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} {best_status}"
                )
            else:
                lr_suffix = f" | lr={optimiser.param_groups[0]['lr']:.2e}" if print_lr else ""
                print(
                    f"  Epoch {epoch:2d}/{epochs} | "
                    f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | "
                    f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}{lr_suffix}"
                )
                if best_status:
                    best_path = self.checkpoint_dir / f"{run_label}_best.pth"
                    print(f"    ✓ New best val_acc={best_val_acc:.4f} — saved to {best_path}")

        trainer.run(train_loader, max_epochs=epochs)

        best_path = self.checkpoint_dir / f"{run_label}_best.pth"
        if best_path.exists():
            model.load_state_dict(torch.load(best_path, map_location=self.device))

        return (history, best_val_acc) if return_best_acc else history

    def train_transfer_model(
        self,
        model,
        model_name,
        train_loader,
        val_loader,
        learning_rate: float,
        epochs: int,
        early_stop_patience: int = 3,
        weight_decay: float = 0.001,
    ):
        """Transfer-learning training loop (Part A)."""
        optimiser = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode="min", factor=0.5, patience=2)

        return self._run_training_loop(
            model=model,
            run_label=model_name,
            train_loader=train_loader,
            val_loader=val_loader,
            optimiser=optimiser,
            scheduler=scheduler,
            epochs=epochs,
            early_stop_patience=early_stop_patience,
            save_last_checkpoint=True,
            show_progress_bar=True,
            print_run_header=model_name,
            return_best_acc=False,
        )

    def fine_tune(
        self,
        model,
        model_name: str,
        n_unfrozen: int,
        train_loader,
        val_loader,
        epochs: int = None,
        early_stop_patience: int = 4,
    ):
        """Fine-tuning loop with differential head/backbone learning rates (Part B)."""
        epochs = epochs or self.default_epochs

        head_attr = "fc" if model_name in ("googlenet", "resnet") else "classifier"
        head_params = list(getattr(model, head_attr).parameters())
        head_ids = {id(p) for p in head_params}
        backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]

        param_groups = [{"params": head_params, "lr": self.head_lr}]
        if backbone_params:
            param_groups.append({"params": backbone_params, "lr": self.backbone_lr})

        optimiser = optim.Adam(param_groups, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode="min", factor=0.5, patience=3)

        run_label = f"{model_name}_ft_unfreeze{n_unfrozen}"
        return self._run_training_loop(
            model=model,
            run_label=run_label,
            train_loader=train_loader,
            val_loader=val_loader,
            optimiser=optimiser,
            scheduler=scheduler,
            epochs=epochs,
            early_stop_patience=early_stop_patience,
            save_last_checkpoint=False,
            show_progress_bar=False,
            print_run_header=None,
            print_lr=True,
            return_best_acc=True,
        )

    def evaluate_on_test(
        self,
        model_utils,
        model_name: str,
        n_unfrozen: int,
        test_loader,
        visualizer=None,
        classes=None,
        top_n: int = 20,
        report_top_pairs: bool = True,
        top_k_pairs: int = 1,
    ) -> float:
        """Load best fine-tuned checkpoint, compute test accuracy, and optionally plot confusion matrix."""
        run_label = f"{model_name}_ft_unfreeze{n_unfrozen}"
        ckpt_path = self.checkpoint_dir / f"{run_label}_best.pth"

        m = model_utils.build_model(model_name)
        model_utils.unfreeze_top_n_layers(m, model_name, n_unfrozen)
        m.load_state_dict(torch.load(ckpt_path, map_location=self.device))
        m.eval()

        correct = total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                preds = m(images).argmax(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)

        acc = correct / total
        print(f"[{run_label}]  test accuracy = {acc:.4f} ({100*acc:.2f} %)")

        if visualizer is not None and classes is not None:
            visualizer.plot_confusion_matrix(
                m,
                test_loader,
                classes,
                top_n=top_n,
                report_top_pairs=report_top_pairs,
                top_k_pairs=top_k_pairs,
            )
        return acc
